# 🧠 AFASIA — Inteligência Artificial Pictórica (IAP)
## Kaggle Notebook — Gemma 4 Good Hackathon 2025

**Projeto:** Atlas Topológico da Afasia + Algoritmo JP  
**Autor:** João Pedro Pereira Passos (UFT — Universidade Federal do Tocantins, 2026)  
**Competição:** [Gemma 4 Good Hackathon — Kaggle / Google DeepMind](https://www.kaggle.com/competitions/gemma-4-good)  
**Repositório:** [github.com/joaopedropassostocantins/AFASIA](https://github.com/joaopedropassostocantins/AFASIA)

---

### Inteligência Artificial Pictórica (IAP) — Teoria e Motivação

**"Construir aviões em vez de imitar pássaros."**

A IA convencional imita a linguagem humana — tokeniza texto, aprende distribuições estatísticas e gera respostas probabilísticas. A **IAP** propõe algo fundamentalmente diferente: operar no nível **pré-linguístico** do pensamento, onde o significado existe como **estrutura topológica** antes de qualquer palavra.

### Equação Central: $G \approx I + F$

$$G \approx I + F$$

| Variável | Nome | Descrição |
|----------|------|-----------|
| $G$ | **Dinâmica do Conhecimento** | O estado em transformação — o pensamento em movimento |
| $I$ | **Incerteza** | O espaço topológico atual, incompleto, do usuário |
| $F$ | **Flexibilidade** | A capacidade de encontrar o próximo passo ótimo |

### Aplicação: Afasia

A **afasia** é uma síndrome neurológica adquirida (AVC, trauma) que compromete o acesso ao código linguístico. Pacientes afásicos pensam em conceitos antes de conseguir verbalizá-los — operam no espaço topológico pré-linguístico que a IAP modela.

### Pipeline deste Notebook
```
Pictograma ARASAAC (palavra em pt-BR)
    ↓  [Gemma 2 — Extrator de Embeddings: hidden state da última camada, token BOS]
Embedding semântico ℝⁿ
    ↓  [Motor Topológico IAP: Diagrama de Persistência + Wasserstein]
Distância topológica entre conceitos
    ↓  [Algoritmo JP: Dijkstra regressivo sobre grafo de Wasserstein]
Plano de comunicação: "Fome → Comer → Maçã"
```

🎥 **Demonstração da Interface Web:** `[PLACEHOLDER — youtube.com/watch?v=SEU_VIDEO_AQUI]`

In [ ]:
# =============================================================================
# CÉLULA 1 — SETUP & INSTALAÇÕES
# Instala e importa todas as dependências para execução no Kaggle (GPU T4 x2)
# =============================================================================

import subprocess, sys, os, math, warnings, unicodedata, heapq
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional

def pip_install(package):
    """Instala um pacote via pip de forma silenciosa."""
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", package],
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )

print("📦 Instalando dependências...")
pip_install("gudhi")       # Persistent homology, diagramas de persistência
pip_install("giotto-tda")  # Pipeline TDA sobre arrays numpy
pip_install("keras>=3.0")  # Keras 3 multi-backend
pip_install("keras-nlp")   # KerasNLP com suporte a Gemma 2
pip_install("networkx")    # Grafo de dependências (Algoritmo JP)
pip_install("scipy")       # Transporte ótimo (Wasserstein exato)
print("✅ Dependências instaladas!")

# Configura backend Keras para JAX (melhor desempenho em GPU T4)
os.environ["KERAS_BACKEND"] = "jax"

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

warnings.filterwarnings("ignore")
print(f"🔧 Ambiente: Python {sys.version.split()[0]}, NumPy {np.__version__}")
print(f"   Keras backend: {os.environ.get('KERAS_BACKEND', 'tensorflow')}")

In [ ]:
# =============================================================================
# CÉLULA 2 — GEMMA 2 COMO EXTRATOR DE EMBEDDINGS
#
# O Gemma 2 é usado como extrator semântico — capturamos o hidden state
# da ÚLTIMA CAMADA no token BOS (equivalente ao [CLS] do BERT) para cada
# rótulo de pictograma em português, gerando o espaço vetorial que
# alimenta o Motor Topológico IAP.
#
# No Kaggle com token configurado: carrega gemma2_2b_en real (float16)
# Sem token:                       usa embedding simulado determinístico
#
# Referência: Algoritmo JP — J.P. Passos, UFT 2026
# =============================================================================

import keras_nlp
import keras

GEMMA_LOADED = False
_gemma_backbone  = None
_gemma_tokenizer = None

print("🤖 Inicializando Gemma 2 (gemma2_2b_en)...")
print("   Modo: extrator de hidden state da última camada (token BOS)")

try:
    _gemma_backbone = keras_nlp.models.GemmaBackbone.from_preset(
        "gemma2_2b_en",
        dtype="float16"   # float16 para economia de VRAM na T4
    )
    _gemma_tokenizer = keras_nlp.models.GemmaTokenizer.from_preset("gemma2_2b_en")
    GEMMA_LOADED = True
    print(f"✅ Gemma 2 carregado! Parâmetros: {_gemma_backbone.count_params():,}")
except Exception as e:
    print(f"⚠️  Gemma 2 indisponível ({type(e).__name__}). Modo simulado ativado.")
    print("   No Kaggle com token configurado, o modelo real será usado.")


# ─── Dicionário de vetores de estado por categoria (IAP) ─────────────────────
# Dimensões: [urgência_vital, valência_emocional, concretude_espacial,
#             relacionalidade, agentividade]
CAT_STATE_VECTORS: Dict[str, List[float]] = {
    'necessidades': [9, 2, 1, 2, 1],
    'sentimentos':  [2, 9, 2, 1, 2],
    'lugares':      [1, 2, 9, 2, 1],
    'pessoas':      [2, 1, 2, 9, 2],
    'acoes':        [1, 2, 1, 2, 9],
    'outros':       [5, 5, 5, 5, 5],
}

# Keywords por categoria — porta fiel do backend TypeScript
ATLAS_CAT_KEYWORDS: Dict[str, List[str]] = {
    'necessidades': ['agua', 'comida', 'comer', 'beber', 'banheiro', 'toalete',
                     'remedio', 'medicina', 'ajuda', 'socorro', 'dormir', 'descanso',
                     'frio', 'calor', 'fome', 'sede', 'higiene', 'banho', 'dente',
                     'roupa', 'sapato', 'dor'],
    'sentimentos':  ['feliz', 'alegre', 'triste', 'dor', 'doer', 'medo', 'ansioso',
                     'ansiedade', 'cansado', 'irritado', 'confuso', 'nervoso', 'raiva',
                     'amor', 'gostar', 'emocao', 'choro', 'chorar', 'rir', 'saudade',
                     'angustia', 'stress', 'depressao'],
    'lugares':      ['casa', 'hospital', 'escola', 'trabalho', 'parque', 'quarto',
                     'rua', 'jardim', 'cidade', 'praia', 'mercado', 'supermercado',
                     'farmacia', 'clinica', 'banheiro'],
    'pessoas':      ['eu', 'mae', 'pai', 'familia', 'medico', 'enfermeiro', 'amigo',
                     'cuidador', 'filho', 'irmao', 'avo', 'pessoa', 'homem', 'mulher',
                     'crianca'],
    'acoes':        ['quero', 'nao', 'sim', 'parar', 'ir', 'vir', 'dar', 'chamar',
                     'ajudar', 'fazer', 'ver', 'ouvir', 'falar', 'andar', 'correr',
                     'sentar', 'levantar', 'brincar', 'trabalhar', 'estudar', 'dormir',
                     'maca', 'maça'],
}


def _normalize_pt(s: str) -> str:
    """Normaliza string PT-BR: minúsculas, remove diacríticos."""
    nfd = unicodedata.normalize('NFD', s.lower())
    return ''.join(c for c in nfd if unicodedata.category(c) != 'Mn')


def inferir_categoria(palavra: str) -> str:
    """
    Infere a categoria semântica IAP de um pictograma pelo rótulo em português.
    Porta fiel de inferCategoria() do backend TypeScript.
    """
    lower = _normalize_pt(palavra)
    for cat, keywords in ATLAS_CAT_KEYWORDS.items():
        norm_kws = [_normalize_pt(k) for k in keywords]
        if any(k in lower or lower in k for k in norm_kws):
            return cat
    return 'outros'


def extrair_embedding_gemma(palavra: str, dimensao: int = 5) -> np.ndarray:
    """
    Extrai o embedding semântico de uma palavra usando Gemma 2.

    Modo Gemma real:
        Tokeniza a palavra e executa forward pass pelo backbone Gemma 2.
        Extrai o hidden state da ÚLTIMA CAMADA no token BOS (posição 0),
        que funciona como representação [CLS] — captura o contexto global
        da sequência. Projeta para `dimensao` via média de blocos.

    Modo simulado (sem Gemma):
        Usa o vetor de estado IAP da categoria inferida + ruído determinístico
        baseado no hash da palavra, garantindo reprodutibilidade.

    Args:
        palavra:  Rótulo do pictograma em português (ex: 'água', 'dor')
        dimensao: Dimensionalidade do vetor de saída (default: 5 — espaço IAP)

    Returns:
        np.ndarray shape (dimensao,) normalizado em [1, 9] — vetor de estado IAP
    """
    if GEMMA_LOADED and _gemma_backbone is not None and _gemma_tokenizer is not None:
        # ── Modo Gemma 2 real ────────────────────────────────────────────────
        # Tokeniza e constrói o dict de entrada para o backbone
        token_ids    = _gemma_tokenizer([palavra])             # (1, seq_len)
        padding_mask = (token_ids != 0).astype("bool")         # (1, seq_len)

        # Forward pass — output: (batch=1, seq_len, hidden_dim)
        hidden_states = _gemma_backbone(
            {"token_ids": token_ids, "padding_mask": padding_mask},
            training=False
        )

        # Extrai representação CLS: hidden state do token BOS (posição 0)
        # Em Gemma, o BOS token (início de sequência) acumula contexto global
        cls_hidden = np.array(hidden_states[0, 0, :], dtype=np.float32)  # (hidden_dim,)

        # Projeta para `dimensao` via média de blocos contíguos
        hidden_dim = cls_hidden.shape[0]
        block_size = max(1, hidden_dim // dimensao)
        truncated  = cls_hidden[:block_size * dimensao]
        raw_5d     = truncated.reshape(dimensao, block_size).mean(axis=1)

    else:
        # ── Modo simulado determinístico ─────────────────────────────────────
        cat  = inferir_categoria(palavra)
        base = np.array(CAT_STATE_VECTORS.get(cat, CAT_STATE_VECTORS['outros']),
                        dtype=np.float32)
        # Ruído determinístico por hash da palavra (garante reprodutibilidade)
        word_hash = sum(ord(c) * (i + 1) for i, c in enumerate(palavra)) % 1000
        rng       = np.random.default_rng(seed=word_hash)
        raw_5d    = base + rng.normal(0, 0.3, dimensao).astype(np.float32)

    # Normaliza para intervalo [1, 9] — compatível com CAT_STATE_VECTORS IAP
    v_min, v_max = raw_5d.min(), raw_5d.max()
    span = max(float(v_max - v_min), 1e-6)
    estado = 1.0 + 8.0 * (raw_5d - v_min) / span

    return estado.astype(np.float32)


# ─── Teste de sanidade ───────────────────────────────────────────────────────
print()
print("🧪 Teste extrair_embedding_gemma():")
for palavra in ['água', 'dor', 'família', 'casa', 'quero', 'maçã']:
    estado = extrair_embedding_gemma(palavra)
    cat    = inferir_categoria(palavra)
    print(f"   '{palavra}' [{cat:12s}] → estado={np.round(estado, 2)}")

print()
print("✅ Extrator de embeddings Gemma 2 pronto!")

In [ ]:
# =============================================================================
# CÉLULA 3 — MOTOR TOPOLÓGICO IAP
#
# Porta fiel do backend TypeScript (artifacts/api-server/src/routes/iap/index.ts):
#   generateSimulatedPersistenceDiagram → gerar_diagrama_persistencia()
#   computeWasserstein                  → calcular_wasserstein()
#   pairwiseTopo                        → distancias_pairwise()
#   classicalMDS                        → mds_classico()
#
# INTEGRAÇÃO COM GEMMA:
#   O estado de entrada de gerar_diagrama_persistencia() é o vetor de 5
#   dimensões retornado por extrair_embedding_gemma() — conectando
#   diretamente: Gemma 2 → Diagrama de Persistência → Wasserstein
#
# Referência: Algoritmo JP — J.P. Passos, UFT 2026
# =============================================================================

@dataclass
class PontoPersistencia:
    """
    Ponto em um Diagrama de Persistência (evento topológico).

    birth    : filtração em que a feature topológica nasce
    death    : filtração em que a feature morre
    dimensao : 0 = componentes conectadas (H0), 1 = buracos (H1)
    lifetime : death - birth (persistência da feature)
    """
    birth: float
    death: float
    dimensao: int
    lifetime: float


def gerar_diagrama_persistencia(
    estado: np.ndarray,
    seed: int
) -> List[PontoPersistencia]:
    """
    Gera um Diagrama de Persistência a partir de um vetor de estado IAP.

    O vetor de estado codifica propriedades pré-linguísticas de um conceito
    (proveniente do Gemma 2 ou do modo simulado). O diagrama captura:
      - H0 (dimensao=0): componentes conectadas — estrutura de clusters
      - H1 (dimensao=1): buracos/loops — complexidade cíclica

    Porta fiel de generateSimulatedPersistenceDiagram() do TypeScript.

    Args:
        estado: Vetor numpy shape (5,) de propriedades semânticas IAP
                (saída de extrair_embedding_gemma() ou CAT_STATE_VECTORS)
        seed:   Semente determinística para reprodutibilidade

    Returns:
        Lista de PontoPersistencia formando o diagrama
    """
    estado_list = list(estado)
    n           = len(estado_list)
    media       = sum(estado_list) / n
    variancia   = sum((v - media) ** 2 for v in estado_list) / n
    pontos      = []

    # ── H0: componentes conectadas (número cresce com intensidade do estado) ─
    num_h0 = max(2, int(media / 3) + 1)
    for i in range(num_h0):
        birth = (i * 0.8 + seed * 0.1) % 2.0
        death = birth + 0.3 + estado_list[i % n] * 0.15
        pontos.append(PontoPersistencia(
            birth=round(birth, 2), death=round(death, 2),
            dimensao=0, lifetime=round(death - birth, 2)
        ))

    # ── H1: buracos topológicos (número cresce com variância / diversidade) ─
    num_h1 = max(1, int(variancia / 5) + 1)
    for i in range(num_h1):
        birth = 0.5 + (i * 0.7 + seed * 0.2) % 1.5
        death = birth + 0.2 + estado_list[(i + 1) % n] * 0.1
        pontos.append(PontoPersistencia(
            birth=round(birth, 2), death=round(death, 2),
            dimensao=1, lifetime=round(death - birth, 2)
        ))

    return pontos


def calcular_wasserstein(
    diag1: List[PontoPersistencia],
    diag2: List[PontoPersistencia]
) -> float:
    """
    Distância de Wasserstein entre dois Diagramas de Persistência.

    Implementa transporte ótimo sobre os pontos H0 (dimensao=0):
    - Pares alinhados:   distância Euclidiana entre os pontos
    - Pontos sem par:    projeção na diagonal = lifetime / √2

    Porta fiel de computeWasserstein() do TypeScript.

    Returns:
        float — distância de Wasserstein (≥ 0, onde 0 = diagramas idênticos)
    """
    h0_1 = [p for p in diag1 if p.dimensao == 0]
    h0_2 = [p for p in diag2 if p.dimensao == 0]

    total = 0.0
    n     = max(len(h0_1), len(h0_2))

    for i in range(n):
        b1 = h0_1[i] if i < len(h0_1) else None
        b2 = h0_2[i] if i < len(h0_2) else None

        if b1 is not None and b2 is not None:
            total += math.sqrt((b1.birth - b2.birth) ** 2 + (b1.death - b2.death) ** 2)
        elif b1 is not None:
            total += b1.lifetime / math.sqrt(2)
        elif b2 is not None:
            total += b2.lifetime / math.sqrt(2)

    return round(total, 4)


def distancias_pairwise(
    pictos: List[Dict],
    wass_scale: float = 3.0
) -> np.ndarray:
    """
    Matriz de Distâncias de Wasserstein para todos os pares de pictogramas.

    INTEGRAÇÃO GEMMA: para cada pictograma chama extrair_embedding_gemma()
    que retorna o estado de 5 dimensões via Gemma 2 (real) ou modo simulado.
    Esse estado alimenta diretamente gerar_diagrama_persistencia().

    Porta fiel de pairwiseTopo() do TypeScript.

    Args:
        pictos:     Lista de dicts {id, palavra, categoria}
        wass_scale: Fator de normalização (default 3.0)

    Returns:
        np.ndarray (N, N) — matriz simétrica, valores em [0, 1]
    """
    n = len(pictos)

    # Extrai estado via Gemma (ou simulado) + ruído determinístico por ID
    state_vectors = []
    for p in pictos:
        # Embedding Gemma → vetor de estado 5D
        base        = extrair_embedding_gemma(p['palavra'], dimensao=5)
        noise_f     = (p['id'] % 7) / 10.0
        sv = np.array([
            max(1.0, float(base[idx]) + noise_f * math.sin(idx * (p['id'] % 13)))
            for idx in range(5)
        ], dtype=np.float32)
        state_vectors.append(sv)

    # Gera diagramas de persistência
    diagramas = [
        gerar_diagrama_persistencia(sv, seed=pictos[i]['id'] % 100)
        for i, sv in enumerate(state_vectors)
    ]

    # Calcula Wasserstein pairwise
    dist = np.zeros((n, n), dtype=np.float32)
    for i in range(n):
        for j in range(i + 1, n):
            w = calcular_wasserstein(diagramas[i], diagramas[j])
            normalized = min(1.0, w / wass_scale)
            dist[i, j] = normalized
            dist[j, i] = normalized

    return dist


def mds_classico(dist: np.ndarray) -> np.ndarray:
    """
    Projeção 2D por MDS Clássico (Multi-Dimensional Scaling).

    Preserva as distâncias de Wasserstein ao projetar N pictogramas
    para ℝ². Usa iteração de potência para os dois maiores autovetores
    da matriz de dupla centração B.

    Porta fiel de classicalMDS() do TypeScript.

    Returns:
        np.ndarray (N, 2) — coordenadas (x, y) do Atlas topológico
    """
    n = dist.shape[0]
    if n == 0: return np.zeros((0, 2))
    if n == 1: return np.zeros((1, 2))
    if n == 2: return np.array([[-dist[0, 1] / 2, 0.0], [dist[0, 1] / 2, 0.0]])

    D2         = dist ** 2
    row_means  = D2.mean(axis=1)
    col_means  = D2.mean(axis=0)
    grand_mean = D2.mean()
    B = -0.5 * (D2 - row_means[:, None] - col_means[None, :] + grand_mean)

    def power_iterate(M: np.ndarray, seed_val: float) -> Tuple[np.ndarray, float]:
        v = np.array([math.sin(i * 1.618 + seed_val) for i in range(n)])
        nm = np.linalg.norm(v)
        v  = v / nm if nm > 1e-12 else np.ones(n) / math.sqrt(n)
        for _ in range(150):
            Mv = M @ v
            nm = np.linalg.norm(Mv)
            v  = Mv / nm if nm > 1e-12 else v
        return v, float(v @ (M @ v))

    v1, lam1 = power_iterate(B, 0.0)
    B2        = B - lam1 * np.outer(v1, v1)   # Deflação: remove 1º autovetor
    v2, lam2  = power_iterate(B2, 1.5)

    s1 = math.sqrt(max(0.0, lam1))
    s2 = math.sqrt(max(0.0, lam2))

    return np.column_stack([np.round(v1 * s1, 3), np.round(v2 * s2, 3)])


# ─── Teste do Motor Topológico ───────────────────────────────────────────────
print("🔬 Teste do Motor Topológico IAP:")
print()
for par in [('água', 'fome'), ('dor', 'feliz'), ('fome', 'maçã')]:
    e1 = extrair_embedding_gemma(par[0])
    e2 = extrair_embedding_gemma(par[1])
    d1 = gerar_diagrama_persistencia(e1, seed=1)
    d2 = gerar_diagrama_persistencia(e2, seed=42)
    w  = calcular_wasserstein(d1, d2)
    print(f"   d_Wasserstein({par[0]!r:8s}, {par[1]!r:8s}) = {w:.4f}")

print()
print("✅ Motor Topológico funcionando com embeddings Gemma 2!")

In [ ]:
# =============================================================================
# CÉLULA 4 — ALGORITMO JP: PLANEJAMENTO REGRESSIVO
#
# Porta fiel de runJpAlgorithmLogic() do backend TypeScript.
# Implementa Dijkstra sobre o grafo topológico de Wasserstein.
#
# A equação G ≈ I + F é operacionalizada aqui:
#   G (dinâmica)      = o caminho planejado pelo algoritmo
#   I (incerteza)     = o estado atual do usuário (pictogramas selecionados)
#   F (flexibilidade) = os 3 próximos passos de menor custo topológico
#
# Pipeline completo: Gemma 2 → Motor Topológico → Dijkstra → Plano
# Referência: Algoritmo JP — J.P. Passos, UFT 2026
# =============================================================================

@dataclass
class PassoPlano:
    """Representa um passo no plano de comunicação do Algoritmo JP."""
    numero_passo:          int
    de_pictograma:         str
    para_pictograma:       str
    distancia_wasserstein: float
    distancia_topologica:  float


@dataclass
class ResultadoJP:
    """Resultado completo do planejamento regressivo pelo Algoritmo JP."""
    sucesso:             bool
    caminho:             List[PassoPlano]
    custo_total:         float
    iteracoes:           int
    mensagem:            str
    vizinhos_imediatos:  List[Dict]


class AlgoritmoJP:
    """
    Motor de Planejamento Regressivo da Inteligência Artificial Pictórica.

    Constrói um grafo topológico onde o peso de cada aresta é a Distância
    de Wasserstein entre os Diagramas de Persistência dos conceitos.
    Os diagramas são gerados a partir dos embeddings do Gemma 2.
    """

    def __init__(self, pictogramas: List[Dict]):
        self.pictogramas      = pictogramas
        self.idx_por_id       = {p['id']: i for i, p in enumerate(pictogramas)}
        self.idx_por_palavra  = {
            _normalize_pt(p['palavra']): i for i, p in enumerate(pictogramas)
        }
        print(f"   🔗 Grafo JP inicializado: {len(pictogramas)} nós")

        # Calcula matriz de distâncias via Gemma 2 → Wasserstein
        print("   🤖 Computando Gemma 2 → embeddings → Wasserstein pairwise...")
        self.dist_matrix = distancias_pairwise(pictogramas)
        print(f"   📐 Matriz de distâncias: {self.dist_matrix.shape}")

        # Projeção 2D (Atlas) via MDS Clássico
        self.coords = mds_classico(self.dist_matrix)
        print(f"   🗺️  Atlas 2D calculado via MDS Clássico")

    def _resolver(self, id_ou_palavra: str) -> Optional[int]:
        """Resolve um ID ou palavra para o índice interno."""
        try:
            return self.idx_por_id.get(int(id_ou_palavra))
        except ValueError:
            return self.idx_por_palavra.get(_normalize_pt(id_ou_palavra))

    def vizinhos_topologicos(self, indice: int, top_k: int = 3) -> List[Dict]:
        """
        Top-k vizinhos mais próximos no espaço de Wasserstein.

        Esta é a operação F (Flexibilidade) na equação G ≈ I + F:
        dados os pictogramas atuais, quais são os próximos passos de menor
        custo topológico?
        """
        n     = len(self.pictogramas)
        pares = sorted([(self.dist_matrix[indice, j], j) for j in range(n) if j != indice])
        return [
            {
                'palavra':                self.pictogramas[j]['palavra'],
                'categoria':              self.pictogramas[j]['categoria'],
                'distancia_wasserstein':  round(float(d), 4),
                'coord_x':                round(float(self.coords[j, 0]), 3),
                'coord_y':                round(float(self.coords[j, 1]), 3),
            }
            for d, j in pares[:top_k]
        ]

    def planejar(self, estado_atual: List[str], objetivo: str) -> ResultadoJP:
        """
        Planejamento regressivo: Dijkstra de estado_atual até objetivo.

        Encontra o caminho de menor custo topológico (Wasserstein) no
        grafo completo de pictogramas, implementando G ≈ I + F.

        Args:
            estado_atual: IDs ou palavras dos pictogramas selecionados (incerteza I)
            objetivo:     ID ou palavra do pictograma alvo

        Returns:
            ResultadoJP com caminho ótimo, custo total e vizinhos imediatos (F)
        """
        n = len(self.pictogramas)

        idx_inicio = None
        for est in estado_atual:
            idx = self._resolver(str(est))
            if idx is not None:
                idx_inicio = idx
                break

        idx_objetivo = self._resolver(str(objetivo))

        if idx_inicio is None or idx_objetivo is None:
            return ResultadoJP(
                sucesso=False, caminho=[], custo_total=0.0, iteracoes=0,
                mensagem=f"Pictograma não encontrado: {estado_atual} / {objetivo}",
                vizinhos_imediatos=[]
            )

        if idx_inicio == idx_objetivo:
            return ResultadoJP(
                sucesso=True, caminho=[], custo_total=0.0, iteracoes=1,
                mensagem="Estado atual já é o objetivo.",
                vizinhos_imediatos=self.vizinhos_topologicos(idx_inicio)
            )

        # ── Dijkstra sobre grafo completo de Wasserstein ─────────────────────
        distancias   = [float('inf')] * n
        distancias[idx_inicio] = 0.0
        predecessores = [None] * n
        heap      = [(0.0, idx_inicio)]
        visitados = set()
        iteracoes = 0

        while heap:
            iteracoes += 1
            custo_atual, u = heapq.heappop(heap)
            if u in visitados:
                continue
            visitados.add(u)
            if u == idx_objetivo:
                break
            for v in range(n):
                if v == u or v in visitados:
                    continue
                custo_aresta = float(self.dist_matrix[u, v])
                novo_custo   = custo_atual + custo_aresta
                if novo_custo < distancias[v]:
                    distancias[v]    = novo_custo
                    predecessores[v] = u
                    heapq.heappush(heap, (novo_custo, v))

        if distancias[idx_objetivo] == float('inf'):
            return ResultadoJP(
                sucesso=False, caminho=[], custo_total=0.0, iteracoes=iteracoes,
                mensagem="Nenhum caminho encontrado.",
                vizinhos_imediatos=self.vizinhos_topologicos(idx_inicio)
            )

        # ── Reconstrução do caminho (planejamento regressivo) ────────────────
        caminho_rev = []
        atual = idx_objetivo
        while atual is not None and atual != idx_inicio:
            prev = predecessores[atual]
            if prev is None:
                break
            caminho_rev.append((prev, atual))
            atual = prev
        caminho_rev.reverse()

        passos = [
            PassoPlano(
                numero_passo=num,
                de_pictograma=self.pictogramas[de]['palavra'],
                para_pictograma=self.pictogramas[para]['palavra'],
                distancia_wasserstein=round(float(self.dist_matrix[de, para]), 4),
                distancia_topologica=round(float(self.dist_matrix[para, idx_objetivo]), 4)
            )
            for num, (de, para) in enumerate(caminho_rev, start=1)
        ]

        p_i = self.pictogramas[idx_inicio]['palavra']
        p_o = self.pictogramas[idx_objetivo]['palavra']
        custo = round(distancias[idx_objetivo], 4)

        return ResultadoJP(
            sucesso=True,
            caminho=passos,
            custo_total=custo,
            iteracoes=iteracoes,
            mensagem=f"Plano: {len(passos)} passo(s) de '{p_i}' a '{p_o}', custo={custo}",
            vizinhos_imediatos=self.vizinhos_topologicos(idx_inicio, top_k=3)
        )


# ─── Conjunto de pictogramas IAP (representativo da base ARASAAC pt-BR) ──────
PICTOGRAMAS_IAP = [
    # Necessidades
    {'id':  1, 'palavra': 'água',     'categoria': 'necessidades'},
    {'id':  2, 'palavra': 'comida',   'categoria': 'necessidades'},
    {'id':  3, 'palavra': 'fome',     'categoria': 'necessidades'},
    {'id':  4, 'palavra': 'sede',     'categoria': 'necessidades'},
    {'id':  5, 'palavra': 'dor',      'categoria': 'necessidades'},
    {'id':  6, 'palavra': 'remédio',  'categoria': 'necessidades'},
    {'id':  7, 'palavra': 'ajuda',    'categoria': 'necessidades'},
    {'id':  8, 'palavra': 'banheiro', 'categoria': 'necessidades'},
    # Sentimentos
    {'id': 11, 'palavra': 'feliz',    'categoria': 'sentimentos'},
    {'id': 12, 'palavra': 'triste',   'categoria': 'sentimentos'},
    {'id': 13, 'palavra': 'medo',     'categoria': 'sentimentos'},
    {'id': 14, 'palavra': 'cansado',  'categoria': 'sentimentos'},
    # Lugares
    {'id': 21, 'palavra': 'casa',     'categoria': 'lugares'},
    {'id': 22, 'palavra': 'hospital', 'categoria': 'lugares'},
    {'id': 23, 'palavra': 'escola',   'categoria': 'lugares'},
    # Pessoas
    {'id': 31, 'palavra': 'eu',       'categoria': 'pessoas'},
    {'id': 32, 'palavra': 'família',  'categoria': 'pessoas'},
    {'id': 33, 'palavra': 'médico',   'categoria': 'pessoas'},
    {'id': 34, 'palavra': 'amigo',    'categoria': 'pessoas'},
    # Ações
    {'id': 41, 'palavra': 'quero',    'categoria': 'acoes'},
    {'id': 42, 'palavra': 'sim',      'categoria': 'acoes'},
    {'id': 43, 'palavra': 'não',      'categoria': 'acoes'},
    {'id': 44, 'palavra': 'comer',    'categoria': 'acoes'},
    {'id': 45, 'palavra': 'ir',       'categoria': 'acoes'},
    {'id': 46, 'palavra': 'maçã',     'categoria': 'acoes'},
]

print("🗺️  Inicializando Algoritmo JP (Gemma 2 → Topologia → Dijkstra)...")
print()
jp = AlgoritmoJP(PICTOGRAMAS_IAP)
print()
print("✅ Algoritmo JP pronto!")

In [ ]:
# =============================================================================
# CÉLULA 5 — SIMULAÇÃO: "FOME → COMER → MAÇÃ"
#
# Demonstra o pipeline completo IAP:
#   1. Paciente com afasia seleciona 'fome' (Estado Atual = I)
#   2. Algoritmo JP traça o caminho até 'maçã' (Objetivo)
#   3. Exibe resultado estruturado: G ≈ I + F
#   4. Visualiza grafo do caminho + Atlas 2D com MDS
#
# Referência: Algoritmo JP — J.P. Passos, UFT 2026
# =============================================================================

ESTADO_ATUAL = ['fome']
OBJETIVO     = 'maçã'

print("=" * 65)
print("  AFASIA — IAP: Simulação da Prova de Conceito")
print("  Algoritmo JP — Planejamento Regressivo")
print("=" * 65)
print(f"\n📍 Estado Atual (I):   {ESTADO_ATUAL}")
print(f"🎯 Estado Objetivo:     {OBJETIVO!r}")
print()

resultado = jp.planejar(estado_atual=ESTADO_ATUAL, objetivo=OBJETIVO)

print("─" * 65)
print(f"  {'✅' if resultado.sucesso else '❌'} {resultado.mensagem}")
print(f"  Custo Total (Wasserstein):  {resultado.custo_total:.4f}")
print(f"  Iterações Dijkstra:         {resultado.iteracoes}")
print("─" * 65)

if resultado.caminho:
    print("\n  📋 G — PLANO DE COMUNICAÇÃO:")
    for passo in resultado.caminho:
        barra = '█' * max(1, int(passo.distancia_wasserstein * 20))
        print(f"\n  Passo {passo.numero_passo}: {passo.de_pictograma!r:15s} "
              f"──[d={passo.distancia_wasserstein:.4f}]──► {passo.para_pictograma!r}")
        print(f"           Dist. até obj.: {passo.distancia_topologica:.4f}  {barra}")

print("\n  🔮 F — FLEXIBILIDADE (próximos 3 passos sugeridos):")
for i, viz in enumerate(resultado.vizinhos_imediatos, 1):
    print(f"  {i}. {viz['palavra']:15s} [{viz['categoria']:12s}]  "
          f"d_Wasserstein={viz['distancia_wasserstein']:.4f}  "
          f"coord=({viz['coord_x']:.3f},{viz['coord_y']:.3f})")

print(f"\n  G ≈ I + F:")
print(f"  G = dinâmica '{ESTADO_ATUAL[0]}' → '{OBJETIVO}' (custo={resultado.custo_total:.4f})")
print(f"  I = incerteza do paciente (estado selecionado: {ESTADO_ATUAL})")
print(f"  F = {len(resultado.vizinhos_imediatos)} próximos passos topológicos sugeridos")
print()


# ─── Visualização Dual: Grafo JP + Atlas 2D MDS ──────────────────────────────
CORES = {
    'necessidades': '#f97316',   # Laranja
    'sentimentos':  '#a855f7',   # Roxo
    'lugares':      '#22c55e',   # Verde
    'pessoas':      '#3b82f6',   # Azul
    'acoes':        '#ef4444',   # Vermelho
    'outros':       '#6b7280',   # Cinza
}

cat_por_palavra = {p['palavra']: p['categoria'] for p in PICTOGRAMAS_IAP}

fig, axes = plt.subplots(1, 2, figsize=(18, 8))
fig.patch.set_facecolor('#0f172a')

# ── Painel 1: Grafo do caminho ótimo (Algoritmo JP) ──────────────────────────
ax1 = axes[0]
ax1.set_facecolor('#1e293b')
ax1.set_title('Algoritmo JP — Caminho Ótimo de Wasserstein',
              color='white', fontsize=13, pad=12)

G_jp = nx.DiGraph()
nomes_caminho = set()

for passo in resultado.caminho:
    G_jp.add_edge(passo.de_pictograma, passo.para_pictograma,
                  weight=passo.distancia_wasserstein)
    nomes_caminho.update([passo.de_pictograma, passo.para_pictograma])

for viz in resultado.vizinhos_imediatos:
    G_jp.add_edge(ESTADO_ATUAL[0], viz['palavra'],
                  weight=viz['distancia_wasserstein'])
    nomes_caminho.add(viz['palavra'])

node_colors = [CORES.get(cat_por_palavra.get(nd, 'outros'), '#6b7280')
               for nd in G_jp.nodes()]
pos = nx.spring_layout(G_jp, seed=42, k=2.0)

arestas_cam = [(p.de_pictograma, p.para_pictograma) for p in resultado.caminho]
arestas_viz = [(ESTADO_ATUAL[0], v['palavra']) for v in resultado.vizinhos_imediatos]

nx.draw_networkx_edges(G_jp, pos, edgelist=arestas_cam, ax=ax1,
                       edge_color='#22d3ee', arrows=True, arrowsize=25,
                       width=2.5, connectionstyle='arc3,rad=0.1')
nx.draw_networkx_edges(G_jp, pos, edgelist=arestas_viz, ax=ax1,
                       edge_color='#4b5563', arrows=True, arrowsize=15,
                       width=1.0, style='dashed', connectionstyle='arc3,rad=0.15')
nx.draw_networkx_nodes(G_jp, pos, node_color=node_colors, node_size=800,
                       ax=ax1, alpha=0.95)
nx.draw_networkx_labels(G_jp, pos, ax=ax1, font_color='white',
                        font_size=9, font_weight='bold')
edge_lbls = {(p.de_pictograma, p.para_pictograma): f"d={p.distancia_wasserstein:.3f}"
             for p in resultado.caminho}
nx.draw_networkx_edge_labels(G_jp, pos, edge_labels=edge_lbls, ax=ax1,
                             font_color='#22d3ee', font_size=7.5)

patches = [mpatches.Patch(color=c, label=cat)
           for cat, c in CORES.items() if cat != 'outros']
ax1.legend(handles=patches, loc='upper right', fontsize=7.5,
           facecolor='#1e293b', edgecolor='#334155', labelcolor='white')
ax1.axis('off')

# ── Painel 2: Atlas 2D — espaço de Wasserstein projetado por MDS ─────────────
ax2 = axes[1]
ax2.set_facecolor('#1e293b')
ax2.set_title('Atlas Topológico IAP — Espaço de Wasserstein (MDS 2D)',
              color='white', fontsize=13, pad=12)

coords = jp.coords
for i, picto in enumerate(PICTOGRAMAS_IAP):
    x, y  = coords[i, 0], coords[i, 1]
    cor   = CORES.get(picto['categoria'], '#6b7280')
    dest  = picto['palavra'] in nomes_caminho
    ax2.scatter(x, y, c=cor, s=120 if dest else 55, alpha=1.0 if dest else 0.5,
                edgecolors='white', linewidths=0.8 if dest else 0.3, zorder=3)
    ax2.annotate(picto['palavra'], (x, y), textcoords='offset points',
                 xytext=(5, 4), fontsize=7.5, color='white',
                 fontweight='bold' if dest else 'normal')

palavra_para_idx = {p['palavra']: i for i, p in enumerate(PICTOGRAMAS_IAP)}
for passo in resultado.caminho:
    id_de   = palavra_para_idx.get(passo.de_pictograma)
    id_para = palavra_para_idx.get(passo.para_pictograma)
    if id_de is not None and id_para is not None:
        x0, y0 = coords[id_de,   0], coords[id_de,   1]
        x1, y1 = coords[id_para, 0], coords[id_para, 1]
        ax2.annotate('', xy=(x1, y1), xytext=(x0, y0),
                     arrowprops=dict(arrowstyle='->', color='#22d3ee',
                                     lw=2.0, mutation_scale=15))
        ax2.annotate(f"d={passo.distancia_wasserstein:.3f}",
                     ((x0 + x1) / 2, (y0 + y1) / 2), fontsize=7,
                     color='#22d3ee', ha='center', zorder=5)

ax2.set_xlabel('Dimensão Topológica 1 (MDS)', color='#94a3b8', fontsize=9)
ax2.set_ylabel('Dimensão Topológica 2 (MDS)', color='#94a3b8', fontsize=9)
ax2.tick_params(colors='#64748b')
for spine in ax2.spines.values():
    spine.set_edgecolor('#334155')
ax2.legend(handles=patches, loc='upper right', fontsize=7.5,
           facecolor='#1e293b', edgecolor='#334155', labelcolor='white')

plt.suptitle(
    f'IAP — Inteligência Artificial Pictórica   |   G ≈ I + F\n'
    f'Gemma 2 → Wasserstein → Dijkstra   |   '
    f'Cenário: "{ESTADO_ATUAL[0]}" → "{OBJETIVO}"   |   '
    f'd_total = {resultado.custo_total:.4f}',
    color='white', fontsize=11, y=1.01
)

plt.tight_layout()
plt.savefig('atlas_topologico_iap.png', dpi=150, bbox_inches='tight',
            facecolor='#0f172a')
plt.show()

print("✅ Visualização salva: atlas_topologico_iap.png")
print()
print("=" * 65)
print("  🧠 A 'sustentação aerodinâmica' está provada:")
print(f"  O sistema navegou de '{ESTADO_ATUAL[0]}' a '{OBJETIVO}'")
print(f"  em {len(resultado.caminho)} passo(s), via geometria topológica pura.")
print("  Sem tokens. Sem probabilidades. Gemma 2 como motor semântico.")
print("=" * 65)